In [1]:
import tensorflow as tf
import random

EMBED_DIM = 128
VOCAB_SIZE = 20000

# ===============================
# 1. UTILS
# ===============================
import re

def chunk_document(text):
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    clean_chunks = []
    for s in sentences:
        s = s.strip()
        # Keep only meaningful sentences
        if len(s.split()) >= 6:
            clean_chunks.append(s)

    return clean_chunks


# ===============================
# 2. TRAINING DOCUMENT
# ===============================
DOCUMENT = """
Retrieval-Augmented Generation (RAG) is a framework that enhances language models by combining information retrieval with natural language generation. Rather than depending entirely on a model’s internal parameters to store and recall knowledge, RAG systems dynamically retrieve relevant information from an external knowledge source at inference time. This retrieved information is then used as contextual grounding for generating responses.

The fundamental motivation behind RAG is that language models, regardless of size, have limited and static knowledge once trained. Updating or expanding this knowledge through retraining is computationally expensive and impractical for many applications. RAG addresses this limitation by decoupling knowledge storage from language understanding, allowing systems to incorporate new or updated information without modifying the core language model.

At a high level, a RAG system consists of two main components: a retriever and a generator. The retriever is responsible for locating relevant pieces of information from a document collection, while the generator uses those retrieved pieces to produce a coherent and contextually grounded response. The effectiveness of a RAG system depends on the tight integration between these two components.

The first stage of a RAG pipeline is document preparation. Raw documents are processed and divided into smaller, manageable units known as chunks. Chunking is necessary because embedding models operate on fixed-length inputs. Each chunk should ideally represent a semantically coherent piece of information, such as a paragraph or a group of related sentences. Poor chunking can degrade retrieval quality, either by fragmenting meaning or by including too much unrelated context.

Once documents are chunked, each chunk is passed through a text encoder to generate a vector embedding. An embedding is a numerical representation of text that captures its semantic meaning. The goal of the encoder is to map texts with similar meanings to nearby points in a high-dimensional vector space. The same encoder is used for both document chunks and user queries to ensure that comparisons are meaningful.

These embeddings are stored in a vector store, which serves as the system’s external memory. Each stored vector is linked to its original text chunk. The vector store enables efficient similarity search, allowing the system to quickly identify which document chunks are most relevant to a given query.

When a user submits a query, the query undergoes the same embedding process as the documents. The resulting query embedding is compared against all stored document embeddings using a similarity function. Common similarity measures include cosine similarity and dot product. These measures quantify how close two vectors are in the embedding space, which correlates with semantic similarity.

The retrieval step selects the top-k document chunks with the highest similarity scores. These chunks are considered the most relevant to the user’s query. Retrieval quality is critical, as irrelevant or noisy chunks can mislead the generation model and result in incorrect or confusing answers.

The retrieved chunks are then passed to the generation component. Instead of answering the query in isolation, the language model receives both the query and the retrieved content as input. This augmented context constrains the model’s output, encouraging it to base its response on retrieved information rather than on memorized or inferred knowledge.

One of the key advantages of RAG is its ability to reduce hallucination. Hallucination occurs when a language model generates information that is factually incorrect or unsupported. By grounding generation in retrieved documents, RAG systems limit the model’s freedom to invent details and improve factual reliability, especially in domain-specific scenarios.

Training plays an important role in improving RAG performance. While some RAG systems rely on pretrained encoders, others fine-tune the retriever using supervised or contrastive learning techniques. A common approach is triplet loss, where the model is trained using a query, a relevant document chunk, and an irrelevant chunk. The objective is to bring the query embedding closer to the relevant chunk while pushing it away from the irrelevant one.

Through training, the retriever learns more precise semantic representations, improving retrieval accuracy. High-quality retrieval directly translates into better generation quality, as the generator receives more relevant and informative context.

Chunk size is a crucial design decision in RAG systems. Smaller chunks tend to improve retrieval precision because they focus on specific ideas, while larger chunks preserve more contextual information. There is no universal optimal chunk size; it depends on the domain, document structure, and query patterns. Many systems adopt overlapping chunks to balance precision and context retention.

Embedding dimensionality is another important consideration. Higher-dimensional embeddings can capture richer semantic information but require more storage and computation. Lower-dimensional embeddings are more efficient but may lose expressive power. RAG systems must strike a balance between retrieval accuracy and resource efficiency.

RAG systems are particularly well-suited for domain-specific knowledge applications. By restricting the document collection to a curated set of trusted sources, the system ensures that responses remain within known boundaries. This makes RAG a strong choice for educational tools, internal documentation assistants, technical support systems, and knowledge-based question answering.

A well-designed RAG system should also handle failure cases gracefully. If retrieval does not return any sufficiently relevant documents, the system should acknowledge that the information is unavailable rather than generating an unsupported answer. This behavior improves user trust and system reliability.

Another strength of RAG is its adaptability. Because knowledge is stored externally, documents can be added, removed, or updated independently of the language model. This enables rapid iteration and continuous improvement without costly retraining cycles.

Evaluation of RAG systems differs from traditional language model evaluation. In addition to generation quality, developers must assess retrieval accuracy, relevance of retrieved chunks, and grounding consistency. Metrics such as recall@k, precision@k, and response faithfulness are commonly used to measure performance.

RAG also supports explainability. Since responses are grounded in retrieved documents, systems can expose the source chunks used to generate an answer. This transparency helps users verify correctness and understand the reasoning behind responses.

Despite its advantages, RAG introduces additional complexity compared to standalone language models. Developers must manage document preprocessing, embedding generation, vector storage, and retrieval tuning. However, this complexity is often justified by the improvements in accuracy, reliability, and controllability.

In summary, Retrieval-Augmented Generation is a powerful paradigm that enhances language models by integrating external knowledge retrieval into the generation process. By separating knowledge storage from reasoning, RAG systems enable scalable, updatable, and reliable question answering. When designed carefully, RAG provides a strong foundation for building intelligent systems that deliver accurate, grounded, and trustworthy responses."""

knowledge_base = chunk_document(DOCUMENT)


# ===============================
# 3. BASE MODEL (INIT)
# ===============================
class TrainableRAG(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.vectorize = tf.keras.layers.TextVectorization(
            max_tokens=VOCAB_SIZE,
            standardize="lower_and_strip_punctuation",
            split="whitespace",
        )

        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(1,), dtype=tf.string),
            self.vectorize,
            tf.keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM),
            tf.keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=1)),
        ])

        self.adapter = tf.keras.layers.Dense(EMBED_DIM, use_bias=False)

    def embed(self, text):
        return self.adapter(self.encoder(text))


model = TrainableRAG()
model.vectorize.adapt(knowledge_base)
_ = model.embed(tf.constant(["init"]))


# ===============================
# 4. TFLITE TRAINER
# ===============================
class TFLiteTrainer(tf.Module):
    def __init__(self, keras_model):
        super().__init__()

        vocab = keras_model.vectorize.get_vocabulary()
        keys = tf.constant(vocab)
        values = tf.range(len(vocab), dtype=tf.int64)
        table_init = tf.lookup.KeyValueTensorInitializer(keys, values)
        self.table = tf.lookup.StaticHashTable(table_init, 0)

        self.embedding_matrix = tf.Variable(
            keras_model.encoder.layers[1].weights[0],
            trainable = True
        )
        self.adapter_matrix = tf.Variable(
            keras_model.adapter.kernel,
            trainable = True
        )

        self.lr = tf.Variable(0.05, dtype=tf.float32, trainable=True)

    def _embed(self, text):
        clean = tf.strings.lower(text)
        tokens = tf.strings.split(tf.expand_dims(clean, 0)).to_tensor("")
        idx = self.table.lookup(tokens)
        vecs = tf.gather(self.embedding_matrix, idx)
        mean = tf.reduce_mean(vecs, axis=1)
        out = tf.matmul(mean, self.adapter_matrix)
        return tf.math.l2_normalize(out[0])

    @tf.function(input_signature=[tf.TensorSpec([], tf.string)])
    def infer(self, text):
        return self._embed(text)

    @tf.function(input_signature=[tf.TensorSpec([EMBED_DIM], tf.float32),
                                  tf.TensorSpec([EMBED_DIM], tf.float32)])
    def score(self, q_emb, d_emb):
        return tf.reduce_sum(q_emb * d_emb)

    @tf.function(input_signature=[
        tf.TensorSpec([], tf.string),
        tf.TensorSpec([], tf.string),
        tf.TensorSpec([], tf.string),
    ])
    def train(self, q, pos, neg):
        with tf.GradientTape() as tape:
            q_emb = self._embed(q)
            p_emb = self._embed(pos)
            n_emb = self._embed(neg)

            loss = tf.maximum(
                0.0,
                0.3 - tf.reduce_sum(q_emb * p_emb) + tf.reduce_sum(q_emb * n_emb)
            )

        grads = tape.gradient(loss, [self.embedding_matrix, self.adapter_matrix])
        for g, v in zip(grads, [self.embedding_matrix, self.adapter_matrix]):
            if g is not None:
                v.assign_sub(self.lr * g)

        return loss


trainer = TFLiteTrainer(model)
CONFIDENCE_THRESHOLD = 0.15


# ===============================
# 5. TOP-K RETRIEVAL
# ===============================
def retrieve_top_k_with_scores(question, k=4):
    q_emb = trainer.infer(tf.constant(question))
    scored = []

    for chunk in knowledge_base:
        c_emb = trainer.infer(tf.constant(chunk))
        score = trainer.score(q_emb,c_emb).numpy()
        scored.append((score, chunk))

    scored.sort(reverse=True)
    return scored[:k]

In [ ]:
print("\n🧠 On-Device Learning Assistant")
print("Type -1 if the answer is correct\n")

while True:
    question = input("\n❓ Question (or exit): ")
    if question.lower() == "exit":
        break

    results = retrieve_top_k_with_scores(question, k=4)

    top_score, top_answer = results[0]
    second_score = results[1][0]

    confidence = top_score - second_score

    # 🟢 HIGH CONFIDENCE → AUTO ACCEPT
    if confidence >= CONFIDENCE_THRESHOLD:
        print("\n🤖 Answer:")
        print(f"{top_answer}")
        print("\n✅ Confident answer (auto-accepted)")
        continue

    # 🔴 LOW CONFIDENCE → ASK USER
    print("\n🤖 Answer:")
    print(f"[0] {top_answer}")

    print("\n💡 Suggestions:")
    for i, (_, text) in enumerate(results[1:], start=1):
        print(f"[{i}] {text}")

    choice = input("\n➡️ Select (-1 = correct, 1-3 = suggestion): ")

    if choice.strip() == "-1":
        print("✅ Answer confirmed. No learning needed.")
        continue

    idx = int(choice)
    positive = results[idx][1]
    negative = top_answer

    loss = trainer.train(
        tf.constant(question),
        tf.constant(positive),
        tf.constant(negative),
    )

    print("🧠 Model updated | Loss:", float(loss))

In [2]:
tf.saved_model.save(
    trainer,
    "rag_saved_model_new",
    signatures={
        "infer": trainer.infer,
        "train": trainer.train,
        "score": trainer.score,
    }
)

In [3]:
import tensorflow as tf
import numpy as np

SAVED_MODEL_DIR = "rag_saved_model_new"
TFLITE_FILE_PATH = "rag_model_new.tflite"

# 1. Initialize Converter from the SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)

# 2. CRITICAL: Enable "Select TF Ops"
# This allows TFLite to use the standard TensorFlow implementation for
# operations like StaticHashTable (Vocabulary lookup) and String ops.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Standard TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS    # Fallback for HashTables/Text
]

# 3. (Optional) Optimization
# This reduces weight size (float32 -> float16) without losing much accuracy
converter.optimizations = [tf.lite.Optimize.DEFAULT]

print("🔄 Converting model... (this may take a moment)")
tflite_model = converter.convert()

# 4. Save the .tflite file
with open(TFLITE_FILE_PATH, "wb") as f:
    f.write(tflite_model)

print(f"✅ Success! Saved to {TFLITE_FILE_PATH}")
print(f"   Size: {len(tflite_model) / 1024:.2f} KB")

🔄 Converting model... (this may take a moment)
✅ Success! Saved to rag_model_new.tflite
   Size: 10107.91 KB
